# Combined Model Evaluation

Compares the explicitly approved final out-of-fold prediction files using discrimination, calibration, Brier score, fixed risk thresholds, top-k capacity metrics, decision curves, and subgroup summaries. Required files are validated before analysis; obsolete prediction files remaining in the Workbench are ignored.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_DIR = Path(
    os.environ.get("LUNG_CANCER_DATA_DIR", Path.home() / "lungcancer")
).expanduser()
RUN_DIR = DATA_DIR / "paper_reproduction"
PRED_DIR = RUN_DIR / "predictions"
FIG_DIR = RUN_DIR / "figures"
for directory in [RUN_DIR, PRED_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

COHORT_PATH = DATA_DIR / "demo_baseline_paper_reproduction.parquet"
if not COHORT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {COHORT_PATH}. Run 01_Rebuild_Paper_Cohort.ipynb first."
    )

cohort = pd.read_parquet(COHORT_PATH).reset_index(drop=True)
cohort["person_id"] = pd.to_numeric(cohort["person_id"], errors="raise").astype("int64")
cohort["has_lungcancer_3Y"] = pd.to_numeric(
    cohort["has_lungcancer_3Y"], errors="raise"
).astype(int)

print("Cohort:", cohort.shape)
print("Cases:", int(cohort["has_lungcancer_3Y"].sum()))

In [ ]:
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import matplotlib.pyplot as plt

# Final manuscript models only. Do not replace this mapping with a wildcard:
# stale or exploratory prediction files may remain in the Workbench.
FINAL_PREDICTION_FILES = {
    "PLCOm2012_published_predictions.parquet": "PLCOm2012_published",
    "PLCOm2012_refitted_logistic_predictions.parquet": "PLCOm2012_refitted_logistic",
    "Clinical_retrained_predictions.parquet": "Clinical_retrained",
    "Demographic_PheCode_predictions.parquet": "Demographic_PheCode",
    "Demographic_PheCode_Smoking_predictions.parquet": "Demographic_PheCode_Smoking",
    "Demographic_CLMBR_predictions.parquet": "Demographic_CLMBR",
    "Demographic_CLMBR_Smoking_predictions.parquet": "Demographic_CLMBR_Smoking",
    "Demographic_Finetuned_CLMBR_predictions.parquet": "Demographic_Finetuned_CLMBR",
    "Demographic_Finetuned_CLMBR_Smoking_predictions.parquet": (
        "Demographic_Finetuned_CLMBR_Smoking"
    ),
}

required_columns = {"person_id", "y_true", "prob", "model"}
missing_files = [
    filename
    for filename in FINAL_PREDICTION_FILES
    if not (PRED_DIR / filename).exists()
]
if missing_files:
    raise FileNotFoundError(
        "Missing required final-model prediction files in "
        f"{PRED_DIR}: {missing_files}. Run the corresponding modeling notebooks first."
    )

prediction_frames = []
for filename, expected_model in FINAL_PREDICTION_FILES.items():
    path = PRED_DIR / filename
    frame = pd.read_parquet(path)

    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(
            f"{filename} is missing required columns: {sorted(missing_columns)}"
        )

    observed_models = set(frame["model"].dropna().astype(str).unique())
    if observed_models != {expected_model}:
        raise ValueError(
            f"{filename} contains model labels {sorted(observed_models)}; "
            f"expected only {expected_model!r}."
        )

    if frame["person_id"].duplicated().any():
        raise ValueError(f"{filename} contains duplicate participant predictions.")

    prediction_frames.append(frame)

predictions = pd.concat(prediction_frames, ignore_index=True)
if predictions.groupby(["model", "person_id"]).size().max() != 1:
    raise ValueError("A model contains duplicate participant predictions.")

ignored_files = sorted(
    path.name
    for path in PRED_DIR.glob("*_predictions.parquet")
    if path.name not in FINAL_PREDICTION_FILES
)
if ignored_files:
    print("Ignoring non-final prediction files:", ignored_files)

print(predictions.groupby("model").agg(
    n=("person_id", "size"),
    cases=("y_true", "sum"),
))


## Overall discrimination, calibration, and error

In [ ]:
def calibration_stats(y, p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, int)
    lp = logit(p).reshape(-1, 1)
    fit = LogisticRegression(C=1e6, solver="lbfgs").fit(lp, y)
    return float(fit.intercept_[0]), float(fit.coef_[0, 0]), float(y.sum() / p.sum())

rows = []
for model, g in predictions.groupby("model"):
    intercept, slope, oe = calibration_stats(g.y_true, g.prob)
    rows.append({
        "model": model, "n": len(g), "cases": int(g.y_true.sum()), "prevalence": g.y_true.mean(),
        "auroc": roc_auc_score(g.y_true, g.prob), "auprc": average_precision_score(g.y_true, g.prob),
        "brier": brier_score_loss(g.y_true, g.prob), "mean_predicted_risk": g.prob.mean(),
        "calibration_intercept": intercept, "calibration_slope": slope, "observed_expected_ratio": oe,
    })
metrics = pd.DataFrame(rows).sort_values("auroc", ascending=False)
display(metrics)
metrics.to_csv(RUN_DIR / "model_metrics.csv", index=False)

In [ ]:
# Participant bootstrap confidence intervals for headline metrics
def bootstrap_metric_ci(g, n_boot=500, seed=42):
    rng = np.random.default_rng(seed)
    y = g["y_true"].to_numpy(int)
    p = g["prob"].to_numpy(float)
    estimates = {"auroc": [], "auprc": [], "brier": []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(g), len(g))
        yb, pb = y[idx], p[idx]
        if np.unique(yb).size < 2:
            continue
        estimates["auroc"].append(roc_auc_score(yb, pb))
        estimates["auprc"].append(average_precision_score(yb, pb))
        estimates["brier"].append(brier_score_loss(yb, pb))
    row = {"model": g["model"].iloc[0]}
    for metric, values in estimates.items():
        point = {
            "auroc": roc_auc_score(y, p),
            "auprc": average_precision_score(y, p),
            "brier": brier_score_loss(y, p),
        }[metric]
        low, high = np.percentile(values, [2.5, 97.5])
        row.update({metric: point, f"{metric}_ci_low": low, f"{metric}_ci_high": high})
    return row

metrics_with_ci = pd.DataFrame([
    bootstrap_metric_ci(g) for _, g in predictions.groupby("model")
])
display(metrics_with_ci)
metrics_with_ci.to_csv(RUN_DIR / "model_metrics_with_bootstrap_ci.csv", index=False)


## Calibration plots

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "k--", label="Ideal")
for model, g in predictions.groupby("model"):
    q = min(10, g.prob.nunique())
    bins = pd.qcut(g.prob, q=q, duplicates="drop")
    cal = g.assign(bin=bins).groupby("bin", observed=True).agg(predicted=("prob", "mean"), observed=("y_true", "mean"))
    ax.plot(cal.predicted, cal.observed, marker="o", label=model)
ax.set(xlabel="Mean predicted risk", ylabel="Observed event rate", title="Out-of-fold calibration")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "calibration.png", dpi=200)
plt.show()

## Fixed risk thresholds

In [ ]:
def threshold_metrics(g, thresholds=(0.005, 0.01, 0.015, 0.02, 0.03)):
    rows = []
    y = g.y_true.to_numpy(int); p = g.prob.to_numpy(float)
    for t in thresholds:
        flag = p >= t
        tp = int((flag & (y == 1)).sum()); fp = int((flag & (y == 0)).sum())
        tn = int((~flag & (y == 0)).sum()); fn = int((~flag & (y == 1)).sum())
        rows.append({"model": g.model.iloc[0], "threshold": t, "flagged_pct": flag.mean(),
                     "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
                     "ppv": tp / max(tp + fp, 1), "npv": tn / max(tn + fn, 1),
                     "cancers_detected": tp, "false_positives_per_cancer": fp / max(tp, 1)})
    return pd.DataFrame(rows)

threshold_table = pd.concat([threshold_metrics(g) for _, g in predictions.groupby("model")], ignore_index=True)
display(threshold_table)
threshold_table.to_csv(RUN_DIR / "risk_threshold_metrics.csv", index=False)

## Top-k capacity analysis

In [ ]:
def topk_metrics(g, fractions=(0.05, 0.10, 0.20)):
    g = g.sort_values("prob", ascending=False).reset_index(drop=True)
    total_cases = int(g.y_true.sum()); rows = []
    for frac in fractions:
        k = max(1, int(np.ceil(len(g) * frac))); selected = g.iloc[:k]
        tp = int(selected.y_true.sum()); fp = k - tp
        rows.append({"model": g.model.iloc[0], "top_fraction": frac, "flagged_n": k,
                     "cancers_detected": tp, "recall_at_k": tp / total_cases,
                     "ppv_at_k": tp / k, "false_positives_per_cancer": fp / max(tp, 1),
                     "implied_probability_cutoff": selected.prob.min()})
    return pd.DataFrame(rows)

topk_table = pd.concat([topk_metrics(g) for _, g in predictions.groupby("model")], ignore_index=True)
display(topk_table)
topk_table.to_csv(RUN_DIR / "topk_metrics.csv", index=False)

## Decision curves

In [ ]:
thresholds = np.linspace(0.0025, 0.05, 100)
fig, ax = plt.subplots(figsize=(8, 6))
for model, g in predictions.groupby("model"):
    y = g.y_true.to_numpy(int); p = g.prob.to_numpy(float); n = len(g)
    net = []
    for t in thresholds:
        flag = p >= t; tp = (flag & (y == 1)).sum(); fp = (flag & (y == 0)).sum()
        net.append(tp / n - fp / n * t / (1 - t))
    ax.plot(thresholds, net, label=model)
prev = predictions.drop_duplicates("person_id").y_true.mean()
ax.plot(thresholds, np.zeros_like(thresholds), "k--", label="Screen none")
ax.plot(thresholds, prev - (1 - prev) * thresholds / (1 - thresholds), color="gray", linestyle=":", label="Screen all")
ax.set(xlabel="Three-year risk threshold", ylabel="Net benefit", title="Decision-curve analysis")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(FIG_DIR / "decision_curve.png", dpi=200)
plt.show()

## Subgroup performance

In [ ]:
subgroup_cols = [c for c in ["sex_at_birth", "race_ethnicity", "smoking_status"] if c in cohort]
base = cohort[["person_id"] + subgroup_cols].copy()
sub = predictions.merge(base, on="person_id", how="left", validate="many_to_one")
rows = []
for model, gm in sub.groupby("model"):
    for variable in subgroup_cols:
        for level, g in gm.groupby(variable, dropna=False):
            if g.y_true.nunique() < 2:
                continue
            rows.append({"model": model, "subgroup_variable": variable, "subgroup": str(level),
                         "n": len(g), "cases": int(g.y_true.sum()),
                         "auroc": roc_auc_score(g.y_true, g.prob),
                         "auprc": average_precision_score(g.y_true, g.prob),
                         "brier": brier_score_loss(g.y_true, g.prob)})
subgroup_metrics = pd.DataFrame(rows)
display(subgroup_metrics)
subgroup_metrics.to_csv(RUN_DIR / "subgroup_metrics.csv", index=False)